# Train ResNet-50 for Glaucoma Stage Classification

Notebook này sử dụng các module dùng chung trong `src/`, train trên split chính thức của `moondream/glaucoma-detection`, chọn checkpoint theo validation loss và chỉ đánh giá test sau khi training hoàn tất.

## 1. Setup

In [ ]:
import json
import os
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'src').exists():
    raise RuntimeError('Không tìm thấy thư mục src. Hãy mở notebook từ repo này.')

os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print('Project root:', PROJECT_ROOT)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import torch
import torch.nn as nn
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    precision_recall_fscore_support,
)

from src.datasets.glaucoma_dataset import IDX_TO_CLASS, build_dataloaders
from src.models.resnet50 import ResNet50Classifier
from src.training.config import (
    CHECKPOINT_DIR,
    DEVICE,
    LEARNING_RATE,
    NUM_CLASSES,
    NUM_EPOCHS,
    RESULTS_FIGURES_DIR,
    RESULTS_METRICS_DIR,
    WEIGHT_DECAY,
    set_seed,
)
from src.training.train_loop import evaluate, fit
import src.training.train_loop as train_loop

if not torch.cuda.is_available():
    raise RuntimeError('CUDA is not available in this notebook kernel')
DEVICE = torch.device('cuda:0')
train_loop.DEVICE = DEVICE

set_seed()
sns.set_theme(style='whitegrid')

print('PyTorch:', torch.__version__)
print('Device:', DEVICE)
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB')

## 2. Configuration

Batch size 16 phù hợp hơn với GPU 8 GB. Nếu vẫn hết VRAM, giảm xuống 8.

In [ ]:
BATCH_SIZE = 16
EPOCHS = NUM_EPOCHS
PATIENCE = 5
MODEL_NAME = 'resnet50'
CLASS_NAMES = [IDX_TO_CLASS[index] for index in range(NUM_CLASSES)]
CHECKPOINT_PATH = Path(CHECKPOINT_DIR) / f'{MODEL_NAME}_best.pt'

print({
    'batch_size': BATCH_SIZE,
    'epochs': EPOCHS,
    'learning_rate': LEARNING_RATE,
    'weight_decay': WEIGHT_DECAY,
    'patience': PATIENCE,
    'classes': CLASS_NAMES,
})

## 3. Load data

In [ ]:
train_loader, val_loader, test_loader = build_dataloaders(batch_size=BATCH_SIZE)

print('Train samples:', len(train_loader.dataset))
print('Validation samples:', len(val_loader.dataset))
print('Test samples:', len(test_loader.dataset))

images, labels = next(iter(train_loader))
print('Image batch:', tuple(images.shape))
print('Label batch:', tuple(labels.shape))
assert images.shape[1:] == (3, 224, 224)
assert labels.min() >= 0 and labels.max() < NUM_CLASSES

## 4. Build model

Baseline hiện tại fine-tune toàn bộ ResNet-50 pretrained bằng AdamW.

In [ ]:
model = ResNet50Classifier(
    num_classes=NUM_CLASSES,
    pretrained=True,
    freeze_features=False,
).to(DEVICE)

loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)

trainable_parameters = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Trainable parameters: {trainable_parameters:,}')

## 5. Train

Để smoke test trước, tạm đặt `EPOCHS = 1` ở phần configuration. Khi chạy ổn, đổi lại thành `NUM_EPOCHS`, restart kernel và chạy lại từ đầu.

In [ ]:
history = fit(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    loss_fn=loss_fn,
    optimizer=optimizer,
    num_epochs=EPOCHS,
    model_name=MODEL_NAME,
    patience=PATIENCE,
)

assert CHECKPOINT_PATH.exists(), f'Không tìm thấy checkpoint: {CHECKPOINT_PATH}'
print('Best checkpoint:', CHECKPOINT_PATH)

## 6. Training curves

In [ ]:
figure_dir = Path(RESULTS_FIGURES_DIR)
figure_dir.mkdir(parents=True, exist_ok=True)

epochs_ran = range(1, len(history['train_loss']) + 1)
plt.figure(figsize=(8, 5))
plt.plot(epochs_ran, history['train_loss'], marker='o', label='Train loss')
plt.plot(epochs_ran, history['val_loss'], marker='o', label='Validation loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('ResNet-50 training history')
plt.legend()
plt.tight_layout()
plt.savefig(figure_dir / 'resnet50_training_curve.png', dpi=160)
plt.show()

## 7. Final test evaluation

Load checkpoint có validation loss thấp nhất trước khi đánh giá test.

In [ ]:
state_dict = torch.load(CHECKPOINT_PATH, map_location=DEVICE, weights_only=True)
model.load_state_dict(state_dict)

test_loss, test_predictions, test_labels = evaluate(model, test_loader, loss_fn)

accuracy = accuracy_score(test_labels, test_predictions)
balanced_accuracy = balanced_accuracy_score(test_labels, test_predictions)
macro_precision, macro_recall, macro_f1, _ = precision_recall_fscore_support(
    test_labels,
    test_predictions,
    labels=range(NUM_CLASSES),
    average='macro',
    zero_division=0,
)

test_metrics = {
    'loss': float(test_loss),
    'accuracy': float(accuracy),
    'balanced_accuracy': float(balanced_accuracy),
    'macro_precision': float(macro_precision),
    'macro_recall': float(macro_recall),
    'macro_f1': float(macro_f1),
}

print(json.dumps(test_metrics, indent=2))
print()
print(classification_report(
    test_labels,
    test_predictions,
    labels=range(NUM_CLASSES),
    target_names=CLASS_NAMES,
    digits=4,
    zero_division=0,
))

## 8. Confusion matrix

In [ ]:
matrix = confusion_matrix(
    test_labels,
    test_predictions,
    labels=range(NUM_CLASSES),
)

plt.figure(figsize=(7, 6))
sns.heatmap(
    matrix,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=CLASS_NAMES,
    yticklabels=CLASS_NAMES,
)
plt.xlabel('Predicted')
plt.ylabel('Ground truth')
plt.title('ResNet-50 test confusion matrix')
plt.tight_layout()
plt.savefig(figure_dir / 'resnet50_confusion_matrix.png', dpi=160)
plt.show()

## 9. Save metrics

In [ ]:
metrics_dir = Path(RESULTS_METRICS_DIR)
metrics_dir.mkdir(parents=True, exist_ok=True)
metrics_path = metrics_dir / 'resnet50.json'

with metrics_path.open('w', encoding='utf-8') as file:
    json.dump(
        {
            'model': MODEL_NAME,
            'class_mapping': {str(index): name for index, name in enumerate(CLASS_NAMES)},
            'batch_size': BATCH_SIZE,
            'epochs_completed': len(history['train_loss']),
            'learning_rate': LEARNING_RATE,
            'weight_decay': WEIGHT_DECAY,
            'metrics': test_metrics,
        },
        file,
        indent=2,
    )

print('Saved metrics:', metrics_path)
print('Saved figures:', figure_dir)
print('Saved checkpoint:', CHECKPOINT_PATH)